### Micrograd

In [14]:
class Value:
    """value in computation graph"""
    
    def __init__(self, data, _children=(), _op='') -> None:
        self.data = data
        self.grad = 0
        
        # for backprop
        self._backward = lambda: None
        self._prev = set(_children)

        self._op = _op

    # calculates gradient for self and other
    # forward pass = out
    # backward pass = out.grad * (local gradient of self or other)

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other) # type checking
        out = Value(self.data + other.data, (self, other), '+') # forward pass

        def _backward(): # adding local derivative is 1. passes gradient
            self.grad += out.grad # backpropped gradient (need += for multiple paths/ancestors/backpropped gradients)
            other.grad += out.grad

        # set output nodes backward function = to our new _backward
        # backprop function is set on forward pass
        out._backward = _backward 

        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def _backward(): # swap derivative
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward

        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only supporting int/float powers for now"
        out = Value(self.data**other, (self,), f'**{other}')

        def _backward(): # power rule of derivatives
            self.grad += (other * self.data**(other-1)) * out.grad
        out._backward = _backward

        return out

    def relu(self):
        out = Value(0 if self.data < 0 else self.data, (self,), 'ReLU')

        def _backward(): 
            self.grad += (out.data > 0) * out.grad # 1 if positive, 0 if not
        out._backward = _backward

        return out

    def backward(self):

        # topological order all of the children in the graph
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v) # appends leaves first that is why we need to reverse? 
        build_topo(self) # builds topological ordered graph from current node backwards

        # go one variable at a time and apply the chain rule to get its gradient
        # not recursive, graph already built
        self.grad = 1
        for v in reversed(topo):
            v._backward() # local gradient * backpropped at v, distributes to v's children

    def __neg__(self): # -self
        return self * -1

    def __radd__(self, other): # other + self
        return self + other

    def __sub__(self, other): # self - other
        return self + (-other)

    def __rsub__(self, other): # other - self
        return other + (-self)

    def __rmul__(self, other): # other * self
        return self * other

    def __truediv__(self, other): # self / other
        return self * other**-1

    def __rtruediv__(self, other): # other / self
        return other * self**-1

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

In [15]:
import random

class Module:

    def zero_grad(self):
        for p in self.parameters():
            p.grad = 0

    def parameters(self):
        return []

class Neuron(Module): # defines # of input connections to a neuron

    def __init__(self, nin, nonlin=True): # nin = # of input values
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(0)
        self.nonlin = nonlin # apply ReLU or not
    
    def __call__(self, x):
        act = sum((wi*xi for wi,xi in zip(self.w, x)), self.b) # dot product of weights and x
        return act.relu() if self.nonlin else act

    def parameters(self):
        return self.w + [self.b]

    def __repr__(self):
        return f"{'ReLU' if self.nonlin else 'Linear'}Neuron({len(self.w)})"

class Layer(Module): # a series of neurons (like hidden state)

    def __init__(self, nin, nout, **kwargs):
        self.neurons = [Neuron(nin, **kwargs) for _ in range(nout)] # nout = # of hidden layer nodes, nin = # of inputs?

    def __call__(self, x):
        out = [n(x) for n in self.neurons]
        return out[0] if len(out) == 1 else out # if final output or not

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

    def __repr__(self):
        return f"Layer of [{', '.join(str(n) for n in self.neurons)}]" 

class MLP(Module):

    def __init__(self, nin, nouts): # ex. MLP(3, [4,4,1]) - input of 3 values. 3 hidden layers with 4, 4, and 1 neurons
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1], nonlin=i!=len(nouts)-1) for i in range(len(nouts))] # stitch layer sizes together

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

    def __repr__(self):
        return f"MLP of [{', '.join(str(layer) for layer in self.layers)}]"


In [ ]:
# testing 

import torch

def test_sanity_check():

    x = Value(-4.0) # our run
    z = 2 * x + 2 + x
    q = z.relu() + z * x
    h = (z * z).relu()
    y = h + q + q * x
    y.backward()
    xmg, ymg = x, y

    x = torch.Tensor([-4.0]).double() # pytorch run
    x.requires_grad = True
    z = 2 * x + 2 + x
    q = z.relu() + z * x
    h = (z * z).relu()
    y = h + q + q * x
    y.backward()
    xpt, ypt = x, y

    # forward pass went well
    assert ymg.data == ypt.data.item()
    # backward pass went well
    assert xmg.grad == xpt.grad.item()

test_sanity_check()

# def test_more_ops():

#     a = Value(-4.0)
#     b = Value(2.0)
#     c = a + b
#     d = a * b + b**3
#     c += c + 1
#     c += 1 + c + (-a)
#     d += d * 2 + (b + a).relu()
#     d += 3 * d + (b - a).relu()
#     e = c - d
#     f = e**2
#     g = f / 2.0
#     g += 10.0 / f
#     g.backward()
#     amg, bmg, gmg = a, b, g

#     a = torch.Tensor([-4.0]).double()
#     b = torch.Tensor([2.0]).double()
#     a.requires_grad = True
#     b.requires_grad = True
#     c = a + b
#     d = a * b + b**3
#     c = c + c + 1
#     c = c + 1 + c + (-a)
#     d = d + d * 2 + (b + a).relu()
#     d = d + 3 * d + (b - a).relu()
#     e = c - d
#     f = e**2
#     g = f / 2.0
#     g = g + 10.0 / f
#     g.backward()
#     apt, bpt, gpt = a, b, g

#     tol = 1e-6
#     # forward pass went well
#     assert abs(gmg.data - gpt.data.item()) < tol
#     # backward pass went well
#     assert abs(amg.grad - apt.grad.item()) < tol
#     assert abs(bmg.grad - bpt.grad.item()) < tol